In [0]:
dbutils.widgets.text("Gold_layer", "gold")
gold = dbutils.widgets.get("Gold_layer")

dbutils.widgets.text("silver_layer", "silver")
silver = dbutils.widgets.get("silver_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

Tables = [t.strip() for t in tables_str.split(",")]

In [0]:
from pyspark.sql.functions import *

In [0]:
gold_schema = f"{gold}.{schema}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

In [0]:
silver_schema = f"{silver}.{schema}"

for i in Tables:
    if i.lower() == "patient":
      patient_df = spark.table(silver_schema+"."+i)
      print(silver_schema+"."+i)
    elif i.lower() == "encounter":
      encounter_df = spark.table(silver_schema+"."+i)
      print(silver_schema+"."+i)
    elif i.lower() == "observation":
      observation_df = spark.table(silver_schema+"."+i)
      print(silver_schema+"."+i)
    elif i.lower() == "condition":
      condition_df = spark.table(silver_schema+"."+i)
      print(silver_schema+"."+i)


patient_df = spark.table("silver.fhir.patient")
encounter_df = spark.table("silver.fhir.encounter")
condition_df = spark.table("silver.fhir.condition")
observation_df = spark.table("silver.fhir.observation")

In [0]:
for i in Tables:
    if i.lower() == "patient":

        dim_patient = "dim_patient"
        gold_patient = f"{gold}.{schema}.{dim_patient}"
        print(gold_patient)

        query = f"""
        CREATE TABLE IF NOT EXISTS {gold_patient} (
            patient_key BIGINT GENERATED ALWAYS AS IDENTITY,
            patient_id STRING,
            active BOOLEAN,
            gender STRING,
            birth_date STRING,
            family_name STRING,
            given_name STRING,
            contact_1 STRING,
            contact_2 STRING,
            city STRING,
            state STRING,
            postal_code STRING,
            country STRING,
            marital_status STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current BOOLEAN
        )
        USING DELTA
        """

        spark.sql(query)

        dim_patient_df = (
            patient_df
            .filter(col("is_current") == True)
            .select(
                "patient_id",
                "active",
                "gender",
                "birth_date",
                "family_name",
                "given_name",
                "contact_1",
                "contact_2",
                "city",
                "state",
                "postal_code",
                "country",
                "marital_status",
                "effective_start_date",
                "effective_end_date",
                "is_current"
            )
        )

        (dim_patient_df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(gold_patient)
        )

        break

else:
    print("Patient table not found in list")

In [0]:
for i in Tables:
    if i.lower() == "encounter":
        
        dim_encounter = "dim_encounter"
        gold_encounter_dim = f"{gold}.{schema}.{dim_encounter}"
        print(gold_encounter_dim)

        query = f"""
        CREATE TABLE IF NOT EXISTS {gold_encounter_dim} (
            encounter_key BIGINT GENERATED ALWAYS AS IDENTITY,
            encounter_id STRING,
            organization_id STRING,
            status STRING,
            period_start STRING,
            period_end STRING
        )
        USING DELTA
        """
        
        spark.sql(query)

        dim_encounter_df = (
            encounter_df
            .filter(col("is_current") == True)
            .select(
                "encounter_id",
                "organization_id",
                "status",
                "period_start",
                "period_end"
            )
        )

        (dim_encounter_df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(gold_encounter_dim)
        )

        break

else:
    print("Encounter table not found in list")

In [0]:
dim_patient = spark.table(gold_patient)
dim_encounter = spark.table(gold_encounter_dim)

In [0]:
for i in Tables:
    if i.lower() == "encounter":
        fact_encounter = "fact_encounter"
        gold_encounter_fact = f"{gold}.{schema}.{fact_encounter}"
        print(gold_encounter_fact)

        query = f"""
        CREATE TABLE IF NOT EXISTS {gold_encounter_fact} (
            encounter_id STRING,
            patient_id STRING,
            status STRING,
            period_start STRING,
            period_end STRING
        )
        USING DELTA
        """
        spark.sql(query)

        fact_encounter_df = (
            encounter_df.alias("s")
            .filter(col("s.is_current") == True)
            .join(
                dim_patient.alias("p"),
                col("s.patient_id") == col("p.patient_id"),
                "inner"
            )
            .join(
                dim_encounter.alias("e"),
                col("s.encounter_id") == col("e.encounter_id"),
                "inner"
            )
            .select(
                col("e.encounter_id"),
                col("p.patient_id"),
                col("s.status"),
                col("s.period_start"),
                col("s.period_end")
            )
        )

        (fact_encounter_df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(gold_encounter_fact)
        )

        break

else:
    print("Encounter table not found in list")

In [0]:
for i in Tables:
    if i.lower() == "condition":
        
        fact_condition = "fact_condition"
        gold_condition = f"{gold}.{schema}.{fact_condition}"
        print(gold_condition)

        query = f"""
        CREATE TABLE IF NOT EXISTS {gold_condition} (
            condition_id STRING,
            patient_id STRING,
            encounter_id STRING,
            condition_text STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current BOOLEAN
        )
        USING DELTA
        """
        spark.sql(query)

        fact_condition_df = (
            condition_df.alias("s")
            .join(
                dim_patient.alias("p"),
                col("s.patient_id") == col("p.patient_id"),
                "inner"
            )
            .join(
                dim_encounter.alias("e"),
                col("s.encounter_id") == col("e.encounter_id"),
                "left"
            )
            .select(
                col("s.condition_id"),
                col("p.patient_id"),
                col("e.encounter_id"),
                col("s.condition_text"),
                col("s.effective_start_date"),
                col("s.effective_end_date"),
                col("s.is_current")
            )
        )

        (fact_condition_df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(gold_condition)
        )

        break

else:
    print("Condition table not found in list")

In [0]:
for i in Tables:
    if i.lower() == "observation":

        fact_observation = "fact_observation"
        gold_observation = f"{gold}.{schema}.{fact_observation}"
        print(gold_observation)

        query = f"""
        CREATE TABLE IF NOT EXISTS {gold_observation} (
            observation_id STRING,
            patient_id STRING,
            observation_text STRING,
            loinc_code STRING,
            category_code STRING,
            value_string STRING,
            value_quantity DOUBLE,
            value_unit STRING,
            effective_datetime TIMESTAMP
        )
        USING DELTA
        """
        spark.sql(query)
        
        fact_observation_df = (
            observation_df.alias("s")
            .filter(col("s.is_current") == True)
            .join(
                dim_patient.alias("p"),
                col("s.patient_id") == col("p.patient_id"),
                "inner"
            )
            .select(
                col("s.observation_id"),
                col("p.patient_id"),
                col("s.observation_text"),
                col("s.loinc_code"),
                col("s.category_code"),
                col("s.value_string"),
                col("s.value_quantity"),
                col("s.value_unit"),
                col("s.effective_datetime")
            )
        )

        (fact_observation_df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(gold_observation)
        )

        break

else:
    print("Observation table not found in list")

In [0]:
%sql
select * from gold.fhir.fact_observation limit 10